# InferSim: calibrate on a real T4 GPU

This notebook measures real LLM inference on Colab's free **T4** GPU and checks InferSim's predictions against it.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**. It takes about 15-20 minutes, most of it downloading two small models.

What it measures, with PyTorch + Hugging Face transformers (FP16, SDPA attention):
- **Prefill** latency for Qwen2.5-0.5B and 1.5B at 128-2,048 prompt tokens.
- **Decode** step latency at batch 1-64 and context 256-2,048 (KV cache cropped back after each step, so every timed step sees the same context).
- **Single GEMMs** (`a @ w`) across the model's real layer shapes, for a direct roofline check.
- **CUDA kernel counts** per step from `torch.profiler`.

Timing uses CUDA events, 3 warmup runs, median of 10. At the end you download `t4_results.zip`; send it back to have it fitted and published.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
!rm -rf infersim && git clone -q --depth 1 https://github.com/Nags-gk/infersim
!pip install -q -e infersim

## Measure (about 10-15 minutes)

In [ ]:
!cd infersim && python calibration/measure.py --out results/t4 --dtype fp16 --warmup 3 --iters 10

## Fit InferSim to the measurements and report the error

Every fourth measurement is held out of the fit; the holdout error is the honest number.

In [ ]:
!cd infersim && infersim calibrate results/t4 --hw t4 --out results/t4/report.json

In [ ]:
import json

with open("infersim/results/t4/report.json") as f:
    r = json.load(f)
print(f"Holdout MAPE: {r['mape_holdout_before']:.1f}% before calibration -> {r['mape_holdout_after']:.1f}% after")
print("Fitted:", {k: round(v, 7) for k, v in r["fitted"].items()})
for k, v in r["by_model_phase"].items():
    print(f"  {k:28s} {v['before']:6.1f}% -> {v['after']:5.1f}%  (n={v['n']})")
if "gemm" in r:
    print(f"GEMM MAPE: {r['gemm']['mape_before']:.1f}% -> {r['gemm']['mape_after']:.1f}%")

## Download the results

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("t4_results", "zip", "infersim/results/t4")
files.download("t4_results.zip")